# Facial Expression (Emotion) Recognition using Deep Learning

Based on: **GeeksforGeeks** style + teacher's **ANN_Implementation** workflow  
Dataset: **FER Emotion Detection** from Kaggle (`ananthu017/emotion-detection-fer`) — folders `train/<emotion>/` and `test/<emotion>/` (e.g. angry, happy, neutral, …)

We implement:
- **Model 1: ANN** (flatten → Dense layers)
- **Model 2: Custom CNN** (Conv2D, MaxPooling, Dropout, Dense)
- **Model 3: VGG16** (transfer learning with ImageNet weights)

Same workflow as the hand-gesture notebook: callbacks, training curves, evaluation, classification report, confusion matrix, sample predictions, and misclassified images.


## Step 1 — Import Libraries

In [ ]:
import os
from glob import glob

import kagglehub
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version     : {np.__version__}")

## Step 2 — Configuration

In [ ]:
import pandas as pd

# Download latest version (FER emotion detection)
path = kagglehub.dataset_download("ananthu017/emotion-detection-fer")
print("Path to dataset files:", path)

DATASET_ROOT = path

# Structure: DATASET_ROOT/train/<emotion>/* and DATASET_ROOT/test/<emotion>/*
image_extensions = ('.png', '.jpg', '.jpeg', '.bmp')
rows = []
for split in ('train', 'test'):
    split_path = os.path.join(DATASET_ROOT, split)
    if not os.path.isdir(split_path):
        continue
    for class_name in sorted(os.listdir(split_path)):
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue
        for fname in os.listdir(class_path):
            if fname.lower().endswith(image_extensions):
                rel_path = os.path.join(split, class_name, fname)
                rows.append({'filename': rel_path, 'class': class_name})

df = pd.DataFrame(rows)
# Use the dataset's own train/ vs test/ split (not random 80/20)
first_seg = df['filename'].astype(str).apply(lambda p: p.replace('\\', '/').split('/')[0])
train_df = df[first_seg == 'train'].reset_index(drop=True)
test_df = df[first_seg == 'test'].reset_index(drop=True)
print(f"Total images: {len(df)}, Classes: {sorted(df['class'].unique())}")
print(f"Training samples: {len(train_df)}, Testing samples: {len(test_df)}")

# For flow_from_dataframe: directory=DATASET_ROOT, relative paths in 'filename'
DATA_DIR = DATASET_ROOT


## Step 2b — Preprocessing: What We Do to Images Before the Model Sees Them

**In simple words:** Raw hand images have pixel values from 0 to 255 and can be different sizes. We need to:
1. **Resize** them to the same size (224×224) so the model always gets the same input.
2. **Rescale** (divide by 255) so values are between 0 and 1 — easier for the model to learn.
3. **Augment** (only for training): slightly shear, zoom, or flip images so the model sees more variety and doesn’t just memorize.

Below you can see one example hand image **before** and **after** these steps. This helps the model learn gesture patterns (palm, fist, OK, etc.) instead of memorizing exact pixels.

In [ ]:
# Get one sample image from training data (any class)
# Make this cell runnable even if Step 3 wasn't run yet (common in Kaggle)
IMG_SIZE = globals().get('IMG_SIZE', 128)

if train_df is None or len(train_df) == 0:
    raise FileNotFoundError('No images found in training dataframe. Run the dataset cell first.')

sample_path = os.path.join(DATA_DIR, train_df['filename'].iloc[0])
img_orig = load_img(sample_path, target_size=(IMG_SIZE, IMG_SIZE), color_mode='rgb')
img_arr_orig = img_to_array(img_orig)

# Rescale: divide by 255 (what the model actually sees)
img_rescaled = img_arr_orig / 255.0

# Augmented versions (same as training)
aug = ImageDataGenerator(rescale=1/255., shear_range=0.2, zoom_range=0.2, horizontal_flip=True)
it = aug.flow(np.expand_dims(img_arr_orig, 0), batch_size=1)
imgs_aug = [next(it)[0] for _ in range(3)]

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
axes[0].imshow(img_orig)
axes[0].set_title(f'1. Original image\n(resized to {IMG_SIZE}×{IMG_SIZE})', fontsize=11)
axes[0].axis('off')
axes[1].imshow(img_rescaled)
axes[1].set_title('2. After rescale\n(values 0–1)', fontsize=11)
axes[1].axis('off')
for i, im in enumerate(imgs_aug):
    axes[2+i].imshow(im)
    axes[2+i].set_title(f'3.{i+1}. Augmented\n(shear/zoom/flip)', fontsize=11)
    axes[2+i].axis('off')
plt.suptitle('Preprocessing steps (in simple words: we resize, rescale, and sometimes slightly change the image so the model learns better)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Step 3 — Data Generators

In [ ]:
# Speed settings
IMG_SIZE = 128  # smaller image = much faster epochs
BATCH_SIZE = 32  # try 64 if you have enough RAM/GPU
IMAGESHAPE = (IMG_SIZE, IMG_SIZE, 3)

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

training_set = train_datagen.flow_from_dataframe(
    train_df,
    directory=DATA_DIR,
    x_col='filename',
    y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=True,
)
test_set = test_datagen.flow_from_dataframe(
    test_df,
    directory=DATA_DIR,
    x_col='filename',
    y_col='class',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=False,
)

# Keras 3+ DataFrameIterator may not have .num_classes — derive from class_indices
class_indices = dict(training_set.class_indices)
num_classes = len(class_indices)
class_names = [name for name, idx in sorted(class_indices.items(), key=lambda x: x[1])]
print(f"Number of classes : {num_classes}")
print(f"Class names      : {class_names}")

# Fix class imbalance: counts from train_df (works even if iterator has no .classes)
train_labels = train_df['class'].map(class_indices).astype(int).values
counts = np.bincount(train_labels, minlength=num_classes)
for i, name in enumerate(class_names):
    print(f"  {name}: {counts[i]} images")
class_weights_arr = compute_class_weight(
    'balanced',
    classes=np.arange(num_classes),
    y=train_labels,
)
class_weight = dict(enumerate(class_weights_arr))
print("Class weights (to balance loss):", class_weight)

## Step 4 — Build Model 1: Custom CNN

A convolutional neural network from scratch: Conv2D → MaxPooling → Dropout → Flatten → Dense.

In [ ]:
tf.random.set_seed(42)

cnn_model = models.Sequential([
    layers.Input(shape=IMAGESHAPE),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax', name='output'),
], name='EmotionFER_CNN')

cnn_model.summary()

## Step 4b — Build Model: ANN (Artificial Neural Network)

Teacher-style ANN: flatten the image to a vector, then Dense layers (no convolutions). Same input shape (224, 224, 3) so we use the same data generators.

In [ ]:
tf.random.set_seed(42)

ann_model = models.Sequential([
    layers.Input(shape=IMAGESHAPE),
    layers.Flatten(),
    layers.Dense(512, activation='relu', name='hidden_1'),
    layers.Dropout(0.3, name='dropout_1'),
    layers.Dense(256, activation='relu', name='hidden_2'),
    layers.Dropout(0.3, name='dropout_2'),
    layers.Dense(128, activation='relu', name='hidden_3'),
    layers.Dense(num_classes, activation='softmax', name='output'),
], name='EmotionFER_ANN')

ann_model.summary()

## Step 5 — Build Model 2: VGG16 (Transfer Learning)

VGG16 with ImageNet weights, frozen layers, and a custom classification head.

In [ ]:
vgg_base = VGG16(
    input_shape=IMAGESHAPE,
    weights='imagenet',
    include_top=False,
)
for layer in vgg_base.layers:
    layer.trainable = False

# A smaller regularized head reduces overfitting
x = layers.GlobalAveragePooling2D()(vgg_base.output)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.4)(x)
prediction = layers.Dense(num_classes, activation='softmax')(x)

vgg_model = Model(inputs=vgg_base.input, outputs=prediction, name='EmotionFER_VGG16')
vgg_model.summary()

## Step 5b — Activation Functions (In Simple Words)

**Why they matter:** The model doesn’t just add and multiply numbers; it passes them through **activation functions** so it can learn non‑linear patterns (like edges, hand shapes, and gesture cues).

- **ReLU (Rectified Linear Unit):** Used in the middle layers. It does: “If the value is negative, turn it to zero; if it’s positive, keep it.” So only “positive evidence” is passed forward.
- **Softmax:** Used only at the **last layer** (output). It turns the model’s raw scores into **probabilities** that add up to 100%. Example: “55% palm, 25% fist, 20% OK”.

**Where we use them:** In our CNN, every Conv2D and Dense (except the last) use **ReLU**. The very last layer uses **Softmax** to output gesture probabilities. The graph below shows how ReLU behaves: negative → 0, positive → unchanged.

In [ ]:
# ReLU: negative values become 0, positive stay the same (storytelling graph)
x = np.linspace(-3, 3, 200)
y_relu = np.maximum(0, x)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, y_relu, 'b-', linewidth=2, label='ReLU output')
ax.axhline(0, color='gray', linestyle='--', alpha=0.7)
ax.axvline(0, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Input value', fontsize=12)
ax.set_ylabel('Output value', fontsize=12)
ax.set_title("ReLU activation: 'If negative → 0; if positive → keep it.' (Used in Conv and hidden layers)", fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Softmax (last layer): turns scores into probabilities that add to 100%, e.g. [0.55 palm, 0.25 fist, 0.20 ok].')

## Step 6 — Train All Three Models (ANN, CNN, VGG16)

We train ANN, CNN, and VGG16 in sequence. Each model's training curves are plotted and saved as a PNG. Results are stored so we can compare and pick the best model at the end.

In [ ]:
# Models to train in order (all three will be trained)
models_to_train = [
    (ann_model, ann_model.name),
    (cnn_model, cnn_model.name),
    (vgg_model, vgg_model.name),
]
all_histories = {}
all_test_metrics = {}
trained_models = {}
print("Will train: ANN, CNN, VGG16. Callbacks and class_weight will be used for each.")

## Step 7 — Set Up Callbacks

**EarlyStopping**: stop if validation loss does not improve.  
**ReduceLROnPlateau**: reduce learning rate when validation loss plateaus.

In [ ]:
# Stop training when validation stops improving (prevents overfitting)
early_stopping = EarlyStopping(
    monitor='val_loss',
    mode='min',
    min_delta=1e-4,
    patience=2,
    restore_best_weights=True,
    verbose=1,
)

# If validation plateaus, reduce LR to try to improve
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    mode='min',
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1,
)

callbacks = [early_stopping, reduce_lr]
print("Callbacks configured: EarlyStopping(val_loss) + ReduceLROnPlateau")

## Step 8 — Train the Model (Loop Over ANN, CNN, VGG16)

Each model is compiled, trained with class_weight and callbacks, evaluated on the test set, then its training curves are plotted with full info and saved as a PNG.

In [ ]:
import os
import json
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger

EPOCHS = 10  # fewer epochs = faster (increase later)

# Speed knob: train on fewer batches per epoch
# - 1.0 = full epoch (slowest)
# - 0.5 = half epoch (about ~2x faster per epoch)
STEPS_FRACTION = 0.5

CHECKPOINT_DIR = 'checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _load_history(path):
    if not os.path.exists(path):
        return None
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return None

def _save_history(path, history_dict):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(history_dict, f, ensure_ascii=False, indent=2)

def train_one_model(base_model):
    # Make checkpoints compatible with current image size
    IMG_SIZE = globals().get('IMG_SIZE', 128)
    tag = f"{base_model.name}_{IMG_SIZE}"

    ckpt_model_path = os.path.join(CHECKPOINT_DIR, f"{tag}.keras")
    ckpt_hist_path = os.path.join(CHECKPOINT_DIR, f"{tag}_history.json")
    ckpt_log_path = os.path.join(CHECKPOINT_DIR, f"{tag}_training_log.csv")

    # Resume: only if checkpoint input size matches current IMG_SIZE
    model = base_model
    if os.path.exists(ckpt_model_path):
        try:
            loaded = load_model(ckpt_model_path)
            expected = (None, IMG_SIZE, IMG_SIZE, 3)
            if tuple(getattr(loaded, 'input_shape', ())) == expected:
                print(f"Resuming from checkpoint: {ckpt_model_path}")
                model = loaded
            else:
                print(f"Checkpoint exists but has different input_shape {loaded.input_shape}; starting fresh for IMG_SIZE={IMG_SIZE}.")
        except Exception as e:
            print(f"Could not load checkpoint {ckpt_model_path} ({e}); starting fresh.")

    prev_hist = _load_history(ckpt_hist_path) or {}
    initial_epoch = len(prev_hist.get('loss', []))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    print(f"Training {model.name} (starting from epoch {initial_epoch})...")

    ckpt_cb = ModelCheckpoint(
        filepath=ckpt_model_path,
        monitor='val_loss',
        mode='min',
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
    )
    log_cb = CSVLogger(ckpt_log_path, append=True)
    callbacks_with_ckpt = [ckpt_cb, log_cb] + list(callbacks)

    try:
        # Use fewer steps per epoch to finish epochs faster
        steps_per_epoch = max(1, int(len(training_set) * STEPS_FRACTION))
        validation_steps = max(1, int(len(test_set) * STEPS_FRACTION))

        test_set.reset()
        history = model.fit(
            training_set,
            epochs=EPOCHS,
            initial_epoch=initial_epoch,
            steps_per_epoch=steps_per_epoch,
            validation_data=test_set,
            validation_steps=validation_steps,
            class_weight=class_weight,
            callbacks=callbacks_with_ckpt,
            verbose=1,
        )
    except KeyboardInterrupt:
        print("Training interrupted by user. Latest completed epoch is already saved in checkpoints.")
        history = None

    if history is not None:
        # Merge history so reruns keep growing the same history file
        merged = dict(prev_hist)
        for k, v in history.history.items():
            merged.setdefault(k, [])
            merged[k].extend(list(v))
        _save_history(ckpt_hist_path, merged)
        all_histories[model.name] = history

    trained_models[model.name] = model

    test_set.reset()
    test_loss, test_acc = model.evaluate(test_set, steps=len(test_set), verbose=0)
    all_test_metrics[model.name] = {'test_loss': float(test_loss), 'test_accuracy': float(test_acc)}

    # Plot curves from merged history if available, otherwise from this run
    hist_for_plot = _load_history(ckpt_hist_path) or (history.history if history is not None else {})
    train_acc = hist_for_plot.get('accuracy', [])
    val_acc = hist_for_plot.get('val_accuracy', [])
    train_loss = hist_for_plot.get('loss', [])
    val_loss = hist_for_plot.get('val_loss', [])

    if train_acc and val_acc and train_loss and val_loss:
        epochs_range = range(1, len(train_acc) + 1)
        final_val_acc = val_acc[-1]
        final_val_loss = val_loss[-1]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(
            f"{model.name} — Final val accuracy: {final_val_acc:.4f}, val loss: {final_val_loss:.4f}",
            fontsize=12,
            fontweight='bold',
        )
        ax1.plot(epochs_range, train_acc, 'b-o', markersize=3, label='Training Accuracy')
        ax1.plot(epochs_range, val_acc, 'r-o', markersize=3, label='Validation Accuracy')
        ax1.set_title('Accuracy (higher = better)')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Accuracy')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax2.plot(epochs_range, train_loss, 'b-o', markersize=3, label='Training Loss')
        ax2.plot(epochs_range, val_loss, 'r-o', markersize=3, label='Validation Loss')
        ax2.set_title('Loss (lower = fewer mistakes)')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{model.name}_training_curves.png', dpi=150, bbox_inches='tight')
        plt.show()

    print(f"  Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")
    print(f"  Checkpoint: {ckpt_model_path}")
    print(f"  History:    {ckpt_hist_path}")

    return model

In [ ]:
# Train ANN only
train_one_model(ann_model)


In [ ]:
# Train CNN only
train_one_model(cnn_model)


In [ ]:
# Train VGG16 only
train_one_model(vgg_model)


## Step 9 — Model Comparison (All Three)

Per-model training curves were plotted and saved above. Below: comparison of validation accuracy and loss for ANN, CNN, and VGG16.
- **Left (Accuracy):** Higher is better. Blue = how well the model fits the training data; red = how well it does on unseen test images. We want both to go up. If blue keeps going up but red stays flat or goes down, the model is **overfitting** (memorizing instead of understanding).
- **Right (Loss):** Lower is better. It’s the “mistake score.” We want both blue and red to go down. When they move together, the model is learning in a healthy way.

In [ ]:
# Per-model training curves are plotted and saved above (Step 8). Comparison graph is below (Step 9b).
pass

## Step 9b — Model Comparison (ANN, CNN, VGG16)

Validation accuracy and loss for all three models. One line per model — easy to see which performed best. Figure is saved as model_comparison.png.

## Step 9c — Select Best Model and Save

Using test accuracy (and test loss as tie-breaker), we select the best of the three models, save it as `best_emotion_model.keras`, and print the reason.

In [ ]:
best_name = max(all_test_metrics, key=lambda k: (all_test_metrics[k]['test_accuracy'], -all_test_metrics[k]['test_loss']))
best_acc = all_test_metrics[best_name]['test_accuracy']
best_loss = all_test_metrics[best_name]['test_loss']
trained_models[best_name].save('best_emotion_model.keras')
print("Results summary (all models):")
for name, m in all_test_metrics.items():
    print(f"  {name}: test accuracy = {m['test_accuracy']:.4f}, test loss = {m['test_loss']:.4f}")
print()
print(f"Best model: {best_name}")
print(f"Reason: Highest test accuracy ({best_acc:.4f}) and lowest test loss ({best_loss:.4f}) among ANN, CNN, and VGG16.")
print("Saved to: best_emotion_model.keras")
model = trained_models[best_name]

In [ ]:
# Plot comparison only if we have at least one history (e.g. from current + previous runs)
if 'all_histories' in dir() and len(all_histories) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Model comparison (ANN, CNN, VGG16): Validation accuracy and loss. Higher acc + lower loss = better.', fontsize=12, fontweight='bold')
    for name, h in all_histories.items():
        ep = range(1, len(h.history['val_accuracy']) + 1)
        ax1.plot(ep, h.history['val_accuracy'], '-o', markersize=4, label=name)
        ax2.plot(ep, h.history['val_loss'], '-o', markersize=4, label=name)
    ax1.set_title('Validation accuracy (higher = better)')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2.set_title('Validation loss (lower = fewer mistakes)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No training history found. Run Step 8 to train all three models.')

## Step 9d — All models: confusion matrices (comparison)

Confusion matrices for **ANN, CNN, and VGG16** on the test set. This lets you compare where each model was right (diagonal) vs wrong (off-diagonal).

In [ ]:
# Get ground truth once (same for all models)
test_set.reset()
y_true_all = test_set.classes

# Predict and compute confusion matrix for each model
all_cms = {}
for name, m in trained_models.items():
    test_set.reset()
    y_pred_probs = m.predict(test_set, steps=len(test_set), verbose=0)
    y_pred_all = np.argmax(y_pred_probs, axis=1)
    all_cms[name] = confusion_matrix(y_true_all, y_pred_all)

# Plot all confusion matrices in one figure (comparison)
fig, axes = plt.subplots(1, len(all_cms), figsize=(5 * max(1, len(all_cms)), 5))
if len(all_cms) == 1:
    axes = [axes]
fig.suptitle('Confusion matrices (test set)', fontsize=14, fontweight='bold')

for idx, (name, cm) in enumerate(all_cms.items()):
    ax = axes[idx]
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=ax,
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=0.5,
    )

    # Add TP/TN/FP/FN labels for binary confusion matrices
    if cm.shape == (2, 2):
        quad_labels = np.array([['TN', 'FP'], ['FN', 'TP']])
        for i in range(2):
            for j in range(2):
                ax.text(
                    j + 0.5,
                    i + 0.5,
                    f"\n{quad_labels[i, j]}",
                    ha='center',
                    va='center',
                    color='black',
                    fontsize=11,
                    fontweight='bold',
                )

    ax.set_title(name)
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('all_models_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9e — Compression comparison (model size vs accuracy)

Compare **model size** (number of parameters) and **test accuracy** for ANN, CNN, and VGG16. Smaller models with good accuracy are more "compressed" (efficient).

In [ ]:
# Model size (parameter count) and test accuracy for each model
names = list(trained_models.keys())
param_counts = [trained_models[n].count_params() for n in names]
accuracies = [all_test_metrics[n]['test_accuracy'] for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Compression comparison: model size (parameters) vs test accuracy', fontsize=13, fontweight='bold')

# Left: parameter count (model size)
bars1 = ax1.bar(names, [p / 1e6 for p in param_counts], color=['#2ecc71', '#3498db', '#9b59b6'], edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Parameters (millions)')
ax1.set_xlabel('Model')
ax1.set_title('Model size (smaller = more compressed)')
ax1.tick_params(axis='x', rotation=15)
for b in bars1:
    h = b.get_height()
    ax1.annotate(f'{h:.2f}M', xy=(b.get_x() + b.get_width()/2, h), ha='center', va='bottom', fontsize=10)

# Right: test accuracy
bars2 = ax2.bar(names, accuracies, color=['#2ecc71', '#3498db', '#9b59b6'], edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Test accuracy')
ax2.set_xlabel('Model')
ax2.set_title('Test accuracy (higher = better)')
ax2.tick_params(axis='x', rotation=15)
ax2.set_ylim(0, 1.05)
for b in bars2:
    h = b.get_height()
    ax2.annotate(f'{h:.2%}', xy=(b.get_x() + b.get_width()/2, h), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('compression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9f — All models: classification report

Precision, recall, and F1-score for **each** model (ANN, CNN, VGG16) on the test set. Compare how each model performs per gesture class (palm, fist, OK, etc.).

In [ ]:
# Classification report for each model
test_set.reset()
y_true_all = test_set.classes
for name, m in trained_models.items():
    test_set.reset()
    y_pred_probs = m.predict(test_set, steps=len(test_set), verbose=0)
    y_pred_all = np.argmax(y_pred_probs, axis=1)
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(classification_report(y_true_all, y_pred_all, target_names=class_names))

## Step 10 — Evaluate on the Test Set (Storytelling)

**What this means:** We run the model on **unseen** test hand images (never used during training). **Test accuracy** = percentage it got right. **Test loss** = how big the mistakes were. Higher accuracy and lower loss = better performance.

In [ ]:
test_set.reset()
test_loss, test_accuracy = model.evaluate(test_set, steps=len(test_set), verbose=1)
print("=" * 50)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")
print("=" * 50)

## Step 11 — Generate Predictions

In [ ]:
print("Classification Report:")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=class_names))

## Step 13 — Confusion Matrix (Storytelling)

**What this graph tells you (in simple words):**
- **Rows = true label** (the real gesture). **Columns = what the model predicted.**
- **Diagonal:** Correct predictions. Bigger numbers here = better.
- **Off-diagonal:** Mistakes (the model confused one gesture with another).

**Note:** For multi-class gesture recognition (10 classes), the confusion matrix is the clearest way to see *which* gestures get mixed up (for example: `01_palm` vs `08_palm_moved`).

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
ax = sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
)

# Add TP/TN/FP/FN labels for binary confusion matrices
if cm.shape == (2, 2):
    quad_labels = np.array([['TN', 'FP'], ['FN', 'TP']])
    for i in range(2):
        for j in range(2):
            ax.text(
                j + 0.5,
                i + 0.5,
                f"\n{quad_labels[i, j]}",
                ha='center',
                va='center',
                color='black',
                fontsize=11,
                fontweight='bold',
            )
else:
    # Multi-class note: diagonal cells are TP for each class
    ax.text(
        0.5,
        -0.12,
        'Diagonal = TP (per class). Off-diagonal = mistakes (confusions).',
        transform=ax.transAxes,
        ha='center',
        va='top',
        fontsize=10,
    )

plt.title('Confusion Matrix: Where the model was right (diagonal) vs wrong (off-diagonal)', fontsize=13, fontweight='bold', pad=15)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

## Step 13b — Graphic report: formulas, values, and results

Below: **TP, TN, FP, FN** from the best model's confusion matrix, then each metric with its **formula**, **given values**, and **result** (Accuracy, Precision, Recall, F1-score, FPR, FNR).

In [ ]:
# Graphic report: key metrics from a multi-class confusion matrix
# Works for gesture recognition (10 classes)

cm = np.array(cm)
num_classes = cm.shape[0]

total = cm.sum()
correct = np.trace(cm)
acc = (correct / total) if total > 0 else 0.0

# Per-class precision/recall/F1 (one-vs-rest from confusion matrix)
per_class = []
for i in range(num_classes):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    per_class.append((i, int(tp), int(fp), int(fn), float(precision), float(recall), float(f1)))

fig, ax = plt.subplots(figsize=(11, 10))
ax.axis('off')
y = 0.98

def line(text, fontsize=11, fontweight='normal'):
    ax.text(
        0.02,
        y,
        text,
        transform=ax.transAxes,
        fontsize=fontsize,
        verticalalignment='top',
        fontweight=fontweight,
        family='monospace',
    )
    return 0.030 if fontsize >= 12 else 0.022

dy = line('Graphic report: metrics from confusion matrix (best model)', fontsize=13, fontweight='bold')
y -= dy

dy = line(f'Overall accuracy = trace(cm) / sum(cm) = {int(correct)} / {int(total)} = {acc:.4f}  ({acc*100:.2f}%)', fontsize=12)
y -= dy

y -= 0.02
dy = line('Per-class (one-vs-rest) metrics:', fontsize=12, fontweight='bold')
y -= dy

y -= 0.01
header = 'idx  class_name               TP    FP    FN   precision  recall   f1'
dy = line(header, fontsize=11, fontweight='bold')
y -= dy

def _safe_name(i):
    try:
        return class_names[i]
    except Exception:
        return str(i)

for i, tp, fp, fn, precision, recall, f1 in per_class:
    name = _safe_name(i)[:20]
    dy = line(f'{i:<3}  {name:<20}  {tp:>4}  {fp:>4}  {fn:>4}    {precision:>7.3f}  {recall:>6.3f}  {f1:>5.3f}', fontsize=11)
    y -= dy
    if y < 0.05:
        break

plt.tight_layout()
plt.savefig('graphic_report_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 14 — Sample Predictions (Storytelling)

**What you see here:** Random hand images from the test set. Each image shows **True** (actual gesture) and **Pred** (what the model predicted). When both match, the model got it right.

In [ ]:
test_set.reset()
x_batch, y_batch = next(test_set)
n_show = min(16, len(x_batch))

pred_batch = model.predict(x_batch[:n_show], verbose=0)
pred_labels = np.argmax(pred_batch, axis=1)
true_labels = np.argmax(y_batch[:n_show], axis=1)

rows, cols = 4, 4
fig, axes = plt.subplots(rows, cols, figsize=(12, 12))
axes = axes.flatten()
for i in range(n_show):
    axes[i].imshow(x_batch[i])
    axes[i].set_title(f"True: {class_names[true_labels[i]]}\nPred: {class_names[pred_labels[i]]}", fontsize=9)
    axes[i].axis('off')
for j in range(n_show, len(axes)):
    axes[j].axis('off')
plt.suptitle('Sample predictions: True = real label, Pred = what the model said (matching = correct)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 15 — Misclassified Images (Storytelling)

**What you see here:** These are hand images where the model **made a mistake** (True ≠ Pred). Looking at these helps us understand which gestures look similar and where the model gets confused.

In [ ]:
wrong_idx = np.where(y_pred != y_true)[0]
print(f"Number of misclassified images: {len(wrong_idx)}")

test_set.reset()
all_images = []
for _ in range(len(test_set)):
    x, _ = next(test_set)
    all_images.append(x)
all_images = np.concatenate(all_images, axis=0)

n_show = min(12, len(wrong_idx))
if n_show == 0:
    print("No misclassified images to show.")
else:
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    axes = axes.flatten()
    for i in range(n_show):
        k = wrong_idx[i]
        axes[i].imshow(all_images[k])
        axes[i].set_title(f"True: {class_names[y_true[k]]}\nPred: {class_names[y_pred[k]]}", fontsize=9)
        axes[i].axis('off')
    for j in range(n_show, len(axes)):
        axes[j].axis('off')
    plt.suptitle('Misclassified: Cases where the model was wrong (helps us see where it gets confused)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Step 16 — Best Model Already Saved

The best model (by test accuracy) was saved in Step 9c as `best_emotion_model.keras`. The cell below confirms the path and reason.

In [ ]:
print("Best model saved in Step 9c: best_emotion_model.keras")
print(f"Model: {best_name}. Reason: highest test accuracy ({best_acc:.4f}), lowest test loss ({best_loss:.4f}).")

## Step 17 — Load Model and Predict on a Single Image

In [ ]:
loaded_model = load_model('best_emotion_model.keras')
# Pick a sample image from test set
sample_row = test_df.iloc[0]
img_path = os.path.join(DATA_DIR, sample_row['filename'])
true_class = sample_row['class']
img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE), color_mode='rgb')
img_arr = img_to_array(img)
img_arr = np.expand_dims(img_arr, axis=0)
if best_name == 'EmotionFER_VGG16':
    img_arr = preprocess_input(img_arr)
else:
    img_arr = img_arr / 255.0

pred = loaded_model.predict(img_arr, verbose=0)
pred_class = np.argmax(pred[0])
print(f'True class: {true_class}')
print(f'Predicted class: {class_names[pred_class]}')
print(f'Predictions: {pred}')